In [ ]:
import warnings
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, train_test_split
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
import xgboost as xgb
import yfinance as yf

warnings.filterwarnings('ignore')

print(
    '--- [MACHINE LEARNING BENCH]: INITIALIZING COMPLETE NON-LINEAR XGBOOST'
    ' PIPELINE ---'
)

# ==============================================================================
# 1. LOCAL DATA INGESTION AND PRE-PROCESSING
# ==============================================================================
df_credit = pd.read_csv(
    'bacen_credito_spread_inadimplencia.csv', sep=';', encoding='latin1'
)
for col in df_credit.columns[1:]:
  df_credit[col] = (
      df_credit[col].astype(str).str.replace(',', '.').astype(float)
  )

month_map = {
    'jan': '01',
    'fev': '02',
    'mar': '03',
    'abr': '04',
    'mai': '05',
    'jun': '06',
    'jul': '07',
    'ago': '08',
    'set': '09',
    'out': '10',
    'nov': '11',
    'dez': '12',
}


def parse_sgs_date(date_str):
  month, year = date_str.split('/')
  return f'20{year}-{month_map[month]}-01'


df_credit['Date_Merge'] = pd.to_datetime(
    df_credit['Data'].apply(parse_sgs_date)
)

df_selic_daily = pd.read_csv(
    'bacen_taxa_selic_diaria.csv', sep=';', encoding='latin1'
)
df_selic_daily['Data'] = pd.to_datetime(
    df_selic_daily['Data'], format='%d/%m/%Y'
)
df_selic_daily.columns = ['Date', 'Selic']
df_selic_daily['Selic'] = (
    df_selic_daily['Selic'].astype(str).str.replace(',', '.').astype(float)
)
df_selic_monthly = (
    df_selic_daily.resample('MS', on='Date').mean().reset_index()
)
df_selic_monthly.columns = ['Date_Merge', 'Selic_Over']

df_target = pd.read_excel('fgv_ice_original.xls', sheet_name='sheet')
df_target['Date_Merge'] = pd.to_datetime(
    df_target['Data']
    .str.split('/')
    .apply(lambda x: f'{x[1]}-{x[0]}-01')
)
df_target.rename(columns={'ICE': 'ICE'}, inplace=True)


def process_daily_investing(file_path, target_col):
  df = pd.read_csv(file_path, encoding='utf-8')
  df['Date_dt'] = pd.to_datetime(df['Data'], format='%d.%m.%Y')
  df['Último'] = (
      df['Último']
      .astype(str)
      .str.replace('.', '', regex=False)
      .str.replace(',', '.', regex=False)
      .astype(float)
  )
  df_m = df.resample('MS', on='Date_dt').last().reset_index()
  return df_m[['Date_dt', 'Último']].rename(
      columns={'Date_dt': 'Date_Merge', 'Último': target_col}
  )


df_cds = process_daily_investing('investing_cds_5y_brasil.csv', 'CDS_5Y_Brazil')
df_long_rate = process_daily_investing(
    'investing_juro_longo_5y.csv', 'Yield_5Y_Brazil'
)
df_short_rate = process_daily_investing(
    'investing_juro_curto_1y.csv', 'Yield_1Y_Brazil'
)

df_curve = pd.merge(df_long_rate, df_short_rate, on='Date_Merge', how='inner')
df_curve['Yield_Curve_Slope'] = (
    df_curve['Yield_5Y_Brazil'] - df_curve['Yield_1Y_Brazil']
)

# ==============================================================================
# 2. MARKET DATA EXTRACTION (YAHOO FINANCE) AND MERGING
# ==============================================================================
start_date, end_date = '2011-03-01', '2026-06-01'
tickers = {
    'Ibovespa': '^BVSP',
    'Global_VIX': '^VIX',
    'USD_BRL_FX': 'BRL=X',
}
df_market_monthly = pd.DataFrame()

print('Fetching market data from Yahoo Finance API...')

for name, ticker in tickers.items():
  data_ticker = yf.download(
      ticker, start=start_date, end=end_date, interval='1d', auto_adjust=True
  )
  close_series = (
      data_ticker.loc[:, ('Close', ticker)]
      if isinstance(data_ticker.columns, pd.MultiIndex)
      else data_ticker['Close']
  )

  if name == 'USD_BRL_FX':
    daily_ret = close_series.pct_change()
    monthly_vol = daily_ret.resample('MS').std() * np.sqrt(252) * 100
    df_market_monthly['FX_Realized_Volatility'] = monthly_vol

  df_market_monthly[name] = close_series.resample('MS').last()

bvsp_data = yf.download(
    '^BVSP', start=start_date, end=end_date, interval='1d'
)
df_market_monthly['B3_Trading_Volume'] = (
    bvsp_data.loc[:, ('Volume', '^BVSP')].resample('MS').sum()
    if isinstance(bvsp_data.columns, pd.MultiIndex)
    else bvsp_data['Volume'].resample('MS').sum()
)

df_market_monthly = df_market_monthly.reset_index()
df_market_monthly = df_market_monthly.rename(
    columns={'Date': 'Date_Merge', 'Data': 'Date_Merge', 'index': 'Date_Merge'}
)
df_market_monthly['Ibovespa_Monthly_Return'] = (
    df_market_monthly['Ibovespa'].pct_change() * 100
)

df_final = pd.merge(df_credit, df_selic_monthly, on='Date_Merge', how='inner')
df_final = pd.merge(
    df_final, df_target[['Date_Merge', 'ICE']], on='Date_Merge', how='inner'
)
df_final = pd.merge(df_final, df_cds, on='Date_Merge', how='inner')
df_final = pd.merge(
    df_final,
    df_curve[['Date_Merge', 'Yield_Curve_Slope']],
    on='Date_Merge',
    how='inner',
)
df_final = (
    pd.merge(df_final, df_market_monthly, on='Date_Merge', how='inner')
    .dropna()
    .reset_index(drop=True)
)

features_raw = [
    c
    for c in df_final.columns
    if c not in ['Data', 'Date_Merge', 'ICE', 'Ibovespa', 'USD_BRL_FX']
]

# ==============================================================================
# 3. STL STRUCTURAL FILTERING (DESEASONALIZATION)
# ==============================================================================
print('Applying STL Structural Filter (period=13)...')

stl_y = STL(df_final['ICE'], period=13, robust=True).fit()
df_final['ICE_SA'] = stl_y.trend + stl_y.resid

for col in features_raw:
  stl_x = STL(df_final[col], period=13, robust=True).fit()
  df_final[col + '_SA'] = stl_x.trend + stl_x.resid

features_sa = [col + '_SA' for col in features_raw]

clean_feature_name_map = {
    '20785 - Spread médio das operações de crédito - Pessoas físicas - Total -'
    ' p.p._SA': 'Banking Spread - Households',
    '20787 - Spread médio das operações de crédito com recursos livres -'
    ' Pessoas jurídicas - Total - p.p._SA': 'Banking Spread - Non-Financial Corps',
    '21082 - Inadimplência da carteira de crédito - Total - %_SA': (
        'Credit System Non-Performing Loans (NPL)'
    ),
    'Selic_Over_SA': 'Selic Target Rate',
    'CDS_5Y_Brazil_SA': 'Brazil 5Y Sovereign CDS',
    'Yield_Curve_Slope_SA': 'Yield Curve Slope',
    'Global_VIX_SA': 'Global CBOE VIX Index',
    'FX_Realized_Volatility_SA': 'FX Realized Volatility',
    'B3_Trading_Volume_SA': 'B3 Financial Trading Volume',
    'Ibovespa_Monthly_Return_SA': 'Ibovespa Monthly Return',
}

# ==============================================================================
# 4. BIC-BASED OPTIMAL LAG SELECTION
# ==============================================================================
print('\n--- ESTIMATING OPTIMAL TIME LAGS (BIC CRITERION) ---')
optimal_lags = {}
dynamic_X_list = []
target_y = df_final['ICE_SA']
common_sample_idx = df_final.index[12:]

for col in features_sa:
  best_lag = 1
  min_bic = float('inf')

  for lag in range(1, 13):
    lagged_series = df_final[col].shift(lag)
    X_temp = sm.add_constant(lagged_series.loc[common_sample_idx])
    y_temp = target_y.loc[common_sample_idx]

    temp_model = sm.OLS(y_temp, X_temp).fit()
    current_bic = temp_model.bic

    if current_bic < min_bic:
      min_bic = current_bic
      best_lag = lag

  optimal_lags[col] = best_lag
  english_feature_name = clean_feature_name_map.get(col, col)
  print(f'-> {english_feature_name:<42} | Optimal Lag: {best_lag} months')

  optimal_series = df_final[col].shift(best_lag)
  optimal_series.name = f'{col}_lag_{best_lag}'
  dynamic_X_list.append(optimal_series)

df_dynamic_X = pd.concat(dynamic_X_list, axis=1)
df_valid_dataset = (
    pd.concat([df_final['Date_Merge'], df_dynamic_X, target_y], axis=1)
    .dropna()
    .reset_index(drop=True)
)

dynamic_features_cols = [c for c in df_dynamic_X.columns]
X_full = df_valid_dataset[dynamic_features_cols]
y_full = df_valid_dataset['ICE_SA']

# ==============================================================================
# 5. LABELS AND STRUCTURAL CONSTRAINTS CONFIGURATION
# ==============================================================================
dynamic_clean_names = []
for v in dynamic_features_cols:
  base_name = v.split('_lag_')[0]
  lag_num = v.split('_lag_')[1]
  clean_name = clean_feature_name_map.get(base_name, base_name)
  dynamic_clean_names.append(f'{clean_name} (t-{lag_num})')

X_full_renamed = X_full.copy()
X_full_renamed.columns = dynamic_clean_names

# Monotonicity constraints: -1 (stress/lower confidence) | +1 (relief/higher confidence)
monotonic_signs = {
    'Banking Spread - Households': -1,
    'Banking Spread - Non-Financial Corps': -1,
    'Credit System Non-Performing Loans (NPL)': -1,
    'Selic Target Rate': -1,
    'Brazil 5Y Sovereign CDS': -1,
    'Yield Curve Slope': 1,
    'Global CBOE VIX Index': -1,
    'FX Realized Volatility': -1,
    'B3 Financial Trading Volume': 1,
    'Ibovespa Monthly Return': 1,
}

monotonic_constraints_list = []
for v in dynamic_features_cols:
  base_name = v.split('_lag_')[0]
  clean_name = clean_feature_name_map.get(base_name, base_name)
  monotonic_constraints_list.append(monotonic_signs.get(clean_name, 0))

block_1_clean_names = []
block_2_clean_names = []
for v in dynamic_features_cols:
  base_name = v.split('_lag_')[0]
  lag_num = v.split('_lag_')[1]
  clean_name = clean_feature_name_map.get(base_name, base_name)
  col_renamed = f'{clean_name} (t-{lag_num})'
  if clean_name in [
      'Selic Target Rate',
      'Brazil 5Y Sovereign CDS',
      'FX Realized Volatility',
      'Global CBOE VIX Index',
  ]:
    block_1_clean_names.append(col_renamed)
  else:
    block_2_clean_names.append(col_renamed)

interaction_constraints_clean = [block_1_clean_names, block_2_clean_names]

# ==============================================================================
# 6. MODEL TUNING, GRID SEARCH AND ABLATION STUDY
# ==============================================================================
print('\n--- CALIBRATING HYPERPARAMETERS VIA GRID SEARCH ---')
X_train_renamed, X_test_renamed, y_train, y_test = train_test_split(
    X_full_renamed, y_full, test_size=0.2, random_state=42
)

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

xgb_base = xgb.XGBRegressor(
    objective='reg:squarederror',
    monotone_constraints=tuple(monotonic_constraints_list),
    interaction_constraints=interaction_constraints_clean,
    random_state=42,
)
grid_xgb = GridSearchCV(
    xgb_base,
    param_grid_xgb,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
).fit(X_train_renamed, y_train)

print('\n' + '=' * 75)
print('RUNNING ABLATION STUDY (5-FOLD TIME SERIES CROSS-VALIDATION)')
print('=' * 75)

tscv = TimeSeriesSplit(n_splits=5)


def evaluate_temporal_cv(model, X_data, y_data):
  rmse_list, mae_list, r2_list = [], [], []
  for train_idx, val_idx in tscv.split(X_data):
    X_tr, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
    y_tr, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]

    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)

    rmse_list.append(np.sqrt(mean_squared_error(y_val, preds)))
    mae_list.append(mean_absolute_error(y_val, preds))
    r2_list.append(r2_score(y_val, preds))

  return np.mean(rmse_list), np.mean(mae_list), np.mean(r2_list)


model_unconstrained = xgb.XGBRegressor(
    **grid_xgb.best_params_, random_state=42
)
rmse_uncon, mae_uncon, r2_uncon = evaluate_temporal_cv(
    model_unconstrained, X_full_renamed, y_full
)

model_constrained = xgb.XGBRegressor(
    **grid_xgb.best_params_,
    monotone_constraints=tuple(monotonic_constraints_list),
    interaction_constraints=interaction_constraints_clean,
    random_state=42,
)
rmse_con, mae_con, r2_con = evaluate_temporal_cv(
    model_constrained, X_full_renamed, y_full
)

df_ablation = pd.DataFrame({
    'Model Specification': [
        'Unconstrained XGBoost (Benchmark)',
        'Constrained XGBoost (Structural Architecture)',
    ],
    'RMSE (CV)': [rmse_uncon, rmse_con],
    'MAE (CV)': [mae_uncon, mae_con],
    'R² (CV)': [r2_uncon, r2_con],
})

print(df_ablation.to_string(index=False))
print('=' * 75)

# ==============================================================================
# 7. FINAL MODEL FITTING AND VISUALIZATIONS EXPORT
# ==============================================================================
model_xgb_final = xgb.XGBRegressor(
    **grid_xgb.best_params_,
    monotone_constraints=tuple(monotonic_constraints_list),
    interaction_constraints=interaction_constraints_clean,
    random_state=42,
)
model_xgb_final.fit(X_train_renamed, y_train)

# 7.1. Figure 1: Relative Feature Importance (Gain)
importances = model_xgb_final.feature_importances_
df_importance = pd.DataFrame(
    {'Feature': dynamic_clean_names, 'Importance': importances}
).sort_values(by='Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(
    df_importance['Feature'],
    df_importance['Importance'],
    color='navy',
    edgecolor='black',
    alpha=0.8,
)
for bar in bars:
  width = bar.get_width()
  ax.text(
      width + 0.005,
      bar.get_y() + bar.get_height() / 2,
      f'{width:.4f}',
      va='center',
      ha='left',
      fontweight='bold',
      fontsize=9,
  )

ax.set_xlabel('Relative Gain (Feature Importance)', fontweight='bold')
ax.set_xlim(0, df_importance['Importance'].max() + 0.05)
ax.grid(True, linestyle=':', alpha=0.3)
plt.tight_layout()
plt.savefig('top_features_xgboost_en.png', dpi=300, bbox_inches='tight')
plt.show()

# 7.2. Figure 2: Mean SHAP Prediction Contribution
print('Computing SHAP values and building directional contribution chart...')
explainer = shap.TreeExplainer(model_xgb_final)
shap_values_cell = explainer(X_test_renamed)

val_shap_matrix = (
    shap_values_cell.values
    if hasattr(shap_values_cell, 'values')
    else shap_values_cell
)
mean_abs_shap = np.mean(np.abs(val_shap_matrix), axis=0)

impact_colors = []
for i in range(val_shap_matrix.shape[1]):
  corr = np.corrcoef(X_test_renamed.iloc[:, i], val_shap_matrix[:, i])[0, 1]
  impact_colors.append('seagreen' if corr > 0 else 'firebrick')

df_shap_custom = pd.DataFrame({
    'Feature': dynamic_clean_names,
    'Mean_SHAP': mean_abs_shap,
    'Color': impact_colors,
}).sort_values(by='Mean_SHAP', ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))
bars_shap = ax.barh(
    df_shap_custom['Feature'],
    df_shap_custom['Mean_SHAP'],
    color=df_shap_custom['Color'],
    edgecolor='black',
    alpha=0.85,
)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1.2)

for bar in bars_shap:
  width = bar.get_width()
  ax.text(
      width + 0.01,
      bar.get_y() + bar.get_height() / 2,
      f'{width:.4f}',
      va='center',
      ha='left',
      fontweight='bold',
      fontsize=9,
      color='black',
  )

ax.set_xlabel(
    'Mean Absolute Contribution to Prediction | mean(|SHAP value|)',
    fontweight='bold',
)
ax.set_xlim(0, df_shap_custom['Mean_SHAP'].max() + 0.1)
ax.grid(True, linestyle=':', alpha=0.3)

patch_red = mpatches.Patch(
    color='firebrick', label='Lowers Confidence (Increases Stress)'
)
patch_green = mpatches.Patch(
    color='seagreen', label='Increases Confidence (Attenuates Stress)'
)
ax.legend(
    handles=[patch_red, patch_green],
    loc='lower right',
    frameon=False,
    fontsize=9.5,
)

plt.tight_layout()
plt.savefig('shap_impact_xgboost_en.png', dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 8. VIX TUPINIQUIM II HISTORICAL SERIES (BASE 100)
# ==============================================================================
df_valid_dataset['XGB_Predicted'] = model_xgb_final.predict(X_full_renamed)

vix_xgb_raw = (
    -1
    * (
        df_valid_dataset['XGB_Predicted']
        - df_valid_dataset['XGB_Predicted'].mean()
    )
    / df_valid_dataset['XGB_Predicted'].std()
)
df_valid_dataset['VIX_Tupiniquim_XGB_Base100'] = 100 + (vix_xgb_raw * 10)

fig, ax = plt.subplots(figsize=(14, 5.2))
codace_recessions = [
    ('2014-03-01', '2016-12-01'),
    ('2020-03-01', '2020-06-01'),
]
for start, end in codace_recessions:
  ax.axvspan(
      pd.to_datetime(start),
      pd.to_datetime(end),
      color='lightgrey',
      alpha=0.6,
      label='Official CODACE/FGV Recession' if start == '2014-03-01' else '',
  )

ax.plot(
    df_valid_dataset['Date_Merge'],
    df_valid_dataset['VIX_Tupiniquim_XGB_Base100'],
    color='darkorange',
    linewidth=2.5,
    label='VIX Tupiniquim Index (XGBoost Pipeline | Mean = 100)',
)
ax.axhline(
    y=100,
    color='black',
    linestyle=':',
    linewidth=1.2,
    label='Historical Benchmark (Mean = 100)',
)

ax.set_xlabel('Years', fontweight='bold')
ax.set_ylabel('Index Points (Mean = 100)', fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.4)
ax.legend(loc='best', frameon=False)

plt.tight_layout()
plt.savefig('vix_tupiniquim_base100_xgboost_en.png', dpi=300, bbox_inches='tight')
plt.show()

# ==============================================================================
# 9. PERFORMANCE METRICS AND DATA EXPORT
# ==============================================================================
preds_xgb = model_xgb_final.predict(X_test_renamed)
rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
mae_xgb = mean_absolute_error(y_test, preds_xgb)

print(f'\n-> Holdout RMSE: {rmse_xgb:.4f} | Holdout MAE: {mae_xgb:.4f}')

df_export_vix_xgb = df_valid_dataset[
    ['Date_Merge', 'VIX_Tupiniquim_XGB_Base100']
].copy()
df_export_vix_xgb.rename(
    columns={
        'Date_Merge': 'Date',
        'VIX_Tupiniquim_XGB_Base100': 'VIX_Tupiniquim_XGBoost',
    },
    inplace=True,
)
df_export_vix_xgb.to_excel(
    'vix_tupiniquim_ii_historical_series.xlsx', index=False
)
print("[SUCCESS]: 'vix_tupiniquim_ii_historical_series.xlsx' exported successfully!")